In [ ]:
#!/usr/bin/env python3
"""
27B QLoRA Training — Colab A100
Track C: 27B dense = FT本命
Based on colab_train_template.py
"""

# ============ VARIABLES (ここだけ変える) ============
BASE_MODEL = "Qwen/Qwen3.5-27B"
CHAMPION_ADAPTER = None  # 27B初回、championなし
DATA_URL = "https://raw.githubusercontent.com/clawdia-saka/qwen-training-data/master/rs_sft_train_v1.jsonl"
OUTPUT_REPO = "tetsugan/qwen3.5-27b-lora-v1"
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
LR = 3e-5       # 27B: 9Bより低めのLR
EPOCHS = 2
BATCH_SIZE = 1   # 27B: メモリ節約
GRAD_ACCUM = 8   # effective batch = 8
MAX_LEN = 1536
# ============ END VARIABLES ============

# ============ FIXED CODE (書き換えるな) ============
import os, json, hashlib, gc, subprocess
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['WANDB_DISABLED'] = 'true'

os.system('pip install -q --upgrade transformers bitsandbytes peft accelerate datasets huggingface_hub')

import torch, transformers, bitsandbytes, peft
print(f'torch={torch.__version__} tf={transformers.__version__} bnb={bitsandbytes.__version__} peft={peft.__version__}')
assert torch.cuda.is_available(), 'NO GPU'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({vram:.1f}GB)')
assert vram >= 35, f'27B QLoRA needs A100 (40GB+), got {vram:.0f}GB'

subprocess.run(['wget', '-q', '-O', '/content/train.jsonl', DATA_URL], check=True)
with open('/content/train.jsonl') as f: n_samples = sum(1 for _ in f)
print(f'Data: {n_samples} samples')

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import PeftModel, get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

gc.collect(); torch.cuda.empty_cache()

# A100: bf16 merge path
print("Mode: 4bit QLoRA (A100, 27B)")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'':0},
    attn_implementation='flash_attention_2',
    trust_remote_code=True
)
print(f'Model loaded: {torch.cuda.memory_allocated(0)/1e9:.2f}GB')

if CHAMPION_ADAPTER:
    model = PeftModel.from_pretrained(model, CHAMPION_ADAPTER)
    model = model.merge_and_unload()
    print(f'Champion merged')

model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
))
model.print_trainable_parameters()

# MANDATORY forward test
b = tok('def fibonacci(n):\n    if n <= 1: return n', return_tensors='pt').to(model.device)
with torch.no_grad(): loss = model(**b, labels=b['input_ids']).loss.item()
assert loss < 20 and not (loss != loss), f'Forward FAILED: loss={loss}'
print(f'Forward OK loss={loss:.4f}')

class DS(Dataset):
    def __init__(self, path, tok, ml):
        self.s = []
        with open(path) as f:
            for l in f:
                try:
                    d = json.loads(l)
                    t = tok.apply_chat_template(d['messages'], tokenize=False, add_generation_prompt=False)
                    e = tok(t, truncation=True, max_length=ml, padding=False)
                    if len(e['input_ids']) > 10: self.s.append(e)
                except: pass
        print(f'Loaded {len(self.s)} samples')
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        s = self.s[i]
        return {'input_ids':s['input_ids'],'attention_mask':s['attention_mask'],'labels':s['input_ids'].copy()}

dataset = DS('/content/train.jsonl', tok, MAX_LEN)
eval_size = max(1, len(dataset) // 10)
train_size = len(dataset) - eval_size
from torch.utils.data import random_split
train_ds, eval_ds = random_split(dataset, [train_size, eval_size])
print(f'Train: {train_size}, Eval: {eval_size}')
with open('/content/train.jsonl','rb') as f: print(f'hash: {hashlib.md5(f.read()).hexdigest()[:12]}')

trainer = Trainer(model=model, args=TrainingArguments(
    output_dir='/content/output',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=10,
    logging_steps=4,
    save_steps=8,
    save_total_limit=3,
    bf16=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to='none', eval_strategy='steps', eval_steps=20,
    gradient_checkpointing=True,
), train_dataset=train_ds, eval_dataset=eval_ds, data_collator=DataCollatorForSeq2Seq(tok, padding=True))

trainer.train()
print('\nTraining complete!')

model.save_pretrained('/content/output/final')
tok.save_pretrained('/content/output/final')
from huggingface_hub import HfApi
api = HfApi()
try: api.create_repo(OUTPUT_REPO, repo_type='model', private=True, token=HF_TOKEN)
except: pass
api.upload_folder(folder_path='/content/output/final', repo_id=OUTPUT_REPO, repo_type='model', token=HF_TOKEN)
print(f'Uploaded to {OUTPUT_REPO}')

del model, trainer, dataset
gc.collect(); torch.cuda.empty_cache()
from google.colab import runtime
runtime.unassign()


